# Layer 2 New — Unsupervised Road Condition Score

Builds a 3-class paved road condition label (`good` / `med` / `low`) **without using the manual validation set for training**.

Combines two approaches:
- **Option 1 — Physics-motivated condition score**: interpretable combination of S1 backscatter (smoothness), temporal stability, S2 bare-soil intrusion, vegetation encroachment.
- **Option 4 — Deterioration trend index**: per-road slopes of VV / BSI / NDVI over 2017–2023 to flag roads getting worse over time.

Tertiles of the combined score → `good` / `med` / `low`. Cross-checked against an unsupervised GaussianMixture clustering (Option 2). Manual validation set used only as held-out evaluation if present.

## 0) Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)

def find_repo_root(start=None, repo_name='Sentinel-FYP'):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if candidate.name == repo_name:
            return candidate
    raise FileNotFoundError(f'Could not find repo root named {repo_name!r}')

ROOT = find_repo_root()
DATA_DIR = ROOT / 'data'
S1_DIR = DATA_DIR / 'ghana_parquet_s1'

pbf_candidates = sorted(DATA_DIR.glob('*.osm.pbf'))
PBF_PATH = pbf_candidates[-1] if pbf_candidates else None

print('Repo root:', ROOT)
print('S1 dir exists:', S1_DIR.exists())
print('PBF:', PBF_PATH.name if PBF_PATH else '(not found)')

## 1) Identify paved roads from OSM

Layer 2 only scores paved roads — Layer 1's policy rule already maps unpaved → high risk. Uses OSM surface tags as the paved label source. In production, swap this for Layer 1 predictions on all 326k roads.

In [ ]:
from pyrosm import OSM

osm = OSM(str(PBF_PATH))
roads_osm = osm.get_network(network_type='driving').copy()

roads_osm['osm_id'] = roads_osm['id'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
roads_osm['surface_clean'] = roads_osm['surface'].astype(str).str.lower().str.strip()
roads_osm['road_name_clean'] = roads_osm.get('name', pd.Series(index=roads_osm.index, dtype='object')).astype(str).str.strip()
roads_osm.loc[roads_osm['road_name_clean'].isin(['', 'nan', 'None']), 'road_name_clean'] = np.nan

focus_classes = ['residential', 'service', 'trunk', 'primary', 'secondary', 'tertiary', 'unclassified']
roads_osm = roads_osm[roads_osm['highway'].isin(focus_classes)].copy()

paved_vals = {'paved', 'asphalt', 'concrete', 'paving_stones', 'concrete:plates', 'sett', 'cobblestone', 'metal', 'bricks', 'cement', 'chipseal'}
unpaved_vals = {'unpaved', 'ground', 'dirt', 'earth', 'gravel', 'fine_gravel', 'sand', 'mud', 'grass', 'compacted', 'pebblestone', 'soil'}

roads_osm['layer1_label'] = np.where(
    roads_osm['surface_clean'].isin(paved_vals), 'paved',
    np.where(roads_osm['surface_clean'].isin(unpaved_vals), 'unpaved', None)
)

# Name propagation
name_known = roads_osm.dropna(subset=['road_name_clean', 'layer1_label']).copy()
name_stats = (
    name_known.groupby('road_name_clean')['layer1_label']
    .agg(lambda s: sorted(set(s.dropna().tolist())))
    .to_frame('classes')
)
name_stats['n_classes'] = name_stats['classes'].apply(len)
name_stats['inferred_label'] = np.where(
    name_stats['n_classes'] == 1,
    name_stats['classes'].apply(lambda x: x[0]),
    np.nan,
)
name_to_label = name_stats['inferred_label'].dropna()

fill_mask = roads_osm['layer1_label'].isna() & roads_osm['road_name_clean'].notna()
roads_osm.loc[fill_mask, 'layer1_label'] = roads_osm.loc[fill_mask, 'road_name_clean'].map(name_to_label)

paved_ids = set(roads_osm.loc[roads_osm['layer1_label'] == 'paved', 'osm_id'].astype(str))
print(f'Paved roads identified: {len(paved_ids)}')

## 2) Load multi-year S1 + S2 features (paved only)

Filters to paved roads at load time to avoid the OOM seen in `layer1_s1.ipynb`.

In [ ]:
idx_cols = ['NDVI', 'NDMI', 'NDBI', 'NDWI', 'BSI']
s1_cols = ['s1_vv_mean', 's1_vh_mean', 's1_vv_minus_vh_mean', 's1_vv_std', 's1_vh_std', 's1_vv_minus_vh_std']
YEARS = [2017, 2018, 2019, 2020, 2021, 2022, 2023]

def read_year(year):
    year_dir = S1_DIR / f'year={year}'
    if not year_dir.exists():
        raise FileNotFoundError(f'Missing: {year_dir}')
    files = sorted(year_dir.glob('*.parquet'))
    frames = [pd.read_parquet(f) for f in files]
    out = pd.concat(frames, ignore_index=True)
    out['year'] = year
    return out

multi_frames = []
for y in YEARS:
    yf = read_year(y)
    yf['osm_id'] = yf['osm_id'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    yf = yf[yf['osm_id'].isin(paved_ids)]
    for c in idx_cols + s1_cols:
        if c in yf.columns:
            yf[c] = pd.to_numeric(yf[c], errors='coerce')
    present_s1 = [c for c in s1_cols if c in yf.columns]
    multi_frames.append(yf[['osm_id', 'quarter', 'year', *idx_cols, *present_s1]])

paved_long = pd.concat(multi_frames, ignore_index=True)
s1_cols_present = [c for c in s1_cols if c in paved_long.columns]

print('Paved-only rows:', len(paved_long))
print('Paved unique roads:', paved_long['osm_id'].nunique())
print('Years:', sorted(paved_long['year'].unique().tolist()))
print('S1 cols:', s1_cols_present)

## 3) Engineer per-road condition features

Per-road aggregates across the full 2017–2023 window:
- **S1 smoothness proxies**: VV mean (low = smoother), VV temporal std (high = unstable), VH mean, VV-VH ratio
- **S2 intrusion proxies**: BSI mean (bare soil on/near road), NDVI mean and seasonal amplitude (vegetation encroachment)
- **Trend slopes** over years: ΔVV/year, ΔBSI/year, ΔNDVI/year — positive slopes indicate deterioration

In [ ]:
# Per-road static aggregates across all quarters+years
agg_kwargs = {
    'vv_mean':       ('s1_vv_mean',           'mean'),
    'vv_std_temp':   ('s1_vv_mean',           'std'),
    'vh_mean':       ('s1_vh_mean',           'mean'),
    'vvvh_mean':     ('s1_vv_minus_vh_mean',  'mean'),
    'vvvh_std':      ('s1_vv_minus_vh_mean',  'std'),
    'ndvi_mean':     ('NDVI',                  'mean'),
    'ndvi_std':      ('NDVI',                  'std'),
    'ndbi_mean':     ('NDBI',                  'mean'),
    'bsi_mean':      ('BSI',                   'mean'),
    'bsi_std':       ('BSI',                   'std'),
    'ndmi_mean':     ('NDMI',                  'mean'),
}
road_feats = paved_long.groupby('osm_id', as_index=False).agg(**agg_kwargs)

# Per-road temporal slope (linear fit across years)
def slope_per_year(df_long, col, out_name):
    yearly = df_long.groupby(['osm_id', 'year'], as_index=False)[col].mean()
    rows = []
    for oid, g in yearly.groupby('osm_id'):
        if len(g) >= 3 and g[col].notna().sum() >= 3:
            x = g['year'].values.astype(float)
            y = g[col].values.astype(float)
            mask = ~np.isnan(y)
            slope = float(np.polyfit(x[mask], y[mask], 1)[0]) if mask.sum() >= 3 else np.nan
        else:
            slope = np.nan
        rows.append({'osm_id': oid, out_name: slope})
    return pd.DataFrame(rows)

vv_slope = slope_per_year(paved_long, 's1_vv_mean', 'vv_slope_year')
bsi_slope = slope_per_year(paved_long, 'BSI', 'bsi_slope_year')
ndvi_slope = slope_per_year(paved_long, 'NDVI', 'ndvi_slope_year')

road_feats = road_feats.merge(vv_slope, on='osm_id', how='left')
road_feats = road_feats.merge(bsi_slope, on='osm_id', how='left')
road_feats = road_feats.merge(ndvi_slope, on='osm_id', how='left')

# Seasonal NDVI amplitude
quarter_map = {'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4}
paved_long['q_num'] = paved_long['quarter'].map(quarter_map)
seasonal = paved_long.groupby(['osm_id', 'q_num'], as_index=False)['NDVI'].mean()
seasonal_p = seasonal.pivot(index='osm_id', columns='q_num', values='NDVI').reset_index()
for q in [1, 2, 3, 4]:
    if q not in seasonal_p.columns:
        seasonal_p[q] = np.nan
seasonal_p['ndvi_seasonal_amp'] = seasonal_p[[1, 2, 3, 4]].max(axis=1) - seasonal_p[[1, 2, 3, 4]].min(axis=1)
road_feats = road_feats.merge(seasonal_p[['osm_id', 'ndvi_seasonal_amp']], on='osm_id', how='left')

# Fill missing with median
num_cols = [c for c in road_feats.columns if c != 'osm_id']
road_feats[num_cols] = road_feats[num_cols].fillna(road_feats[num_cols].median())

print('Road feature table:', road_feats.shape)
road_feats.head()

## 4) Physics-motivated condition score (Option 1)

Higher score = better condition. Robust z-scores (median/MAD) keep outliers from dominating. Weights are deliberate priors — VV smoothness is the strongest single signal, temporal stability next, surface intrusion features as supporting evidence.

In [ ]:
def z_robust(s):
    med = s.median()
    mad = (s - med).abs().median() or 1.0
    return (s - med) / (1.4826 * mad)

road_feats['score_smoothness']   = z_robust(-road_feats['vv_mean'])         # low VV  -> smooth -> good
road_feats['score_stability']    = z_robust(-road_feats['vv_std_temp'])     # low VV temporal std -> stable
road_feats['score_no_intrusion'] = z_robust(-road_feats['bsi_mean'])        # low BSI -> no bare-soil intrusion
road_feats['score_low_veg']      = z_robust(-road_feats['ndvi_seasonal_amp'])  # low amp -> no veg encroachment

road_feats['condition_score'] = (
    1.0 * road_feats['score_smoothness'] +
    0.8 * road_feats['score_stability'] +
    0.5 * road_feats['score_no_intrusion'] +
    0.3 * road_feats['score_low_veg']
)

print(road_feats['condition_score'].describe().round(3))

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(road_feats['condition_score'], bins=80, ax=ax)
ax.set_title('Physics-motivated condition score (higher = better)')
plt.tight_layout()
plt.show()

## 5) Deterioration trend index (Option 4)

Positive value = road is trending worse over 2017–2023. Positive VV slope = surface getting rougher; positive BSI slope = bare soil intruding; positive NDVI slope = vegetation encroaching.

In [ ]:
road_feats['trend_vv']   = z_robust(road_feats['vv_slope_year'])
road_feats['trend_bsi']  = z_robust(road_feats['bsi_slope_year'])
road_feats['trend_ndvi'] = z_robust(road_feats['ndvi_slope_year'])

road_feats['deterioration_index'] = (
    1.0 * road_feats['trend_vv'] +
    0.5 * road_feats['trend_bsi'] +
    0.3 * road_feats['trend_ndvi']
)

print(road_feats['deterioration_index'].describe().round(3))

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(road_feats['deterioration_index'], bins=80, ax=ax)
ax.set_title('Deterioration trend index (positive = trending worse)')
plt.tight_layout()
plt.show()

## 6) Combine into 3-class condition label

`combined_score = condition_score − 0.5 × deterioration_index`. Lower trend penalty so a currently-good road that's slightly trending worse still ranks above a chronically-poor road that's stable. Thresholds at tercile boundaries — the population assumption is roughly balanced, swap to fixed thresholds if domain knowledge says otherwise.

In [ ]:
road_feats['combined_score'] = road_feats['condition_score'] - 0.5 * road_feats['deterioration_index']

q33, q67 = road_feats['combined_score'].quantile([1/3, 2/3])

def classify(x):
    if x >= q67:
        return 'good'
    elif x >= q33:
        return 'med'
    else:
        return 'low'

road_feats['pred_condition'] = road_feats['combined_score'].apply(classify)

print('Threshold q33:', round(q33, 3), '   q67:', round(q67, 3))
print('\nPredicted distribution (%):')
print(road_feats['pred_condition'].value_counts(normalize=True).mul(100).round(1))

fig, ax = plt.subplots(figsize=(8, 4))
for c, color in zip(['good', 'med', 'low'], ['#2ca02c', '#ff7f0e', '#d62728']):
    sns.histplot(road_feats.loc[road_feats['pred_condition'] == c, 'combined_score'],
                 bins=50, alpha=0.5, label=c, color=color, ax=ax)
ax.axvline(q33, color='k', ls='--', alpha=0.4)
ax.axvline(q67, color='k', ls='--', alpha=0.4)
ax.legend()
ax.set_title('Combined score by predicted class')
plt.tight_layout()
plt.show()

## 7) Unsupervised cross-check (Option 2)

Independent GaussianMixture on the raw temporal features. Cluster centroids are sorted by mean VV — lowest VV cluster gets `good`. Agreement with the physics score gives a confidence signal: clusters disagreeing with the score are the harder/ambiguous cases.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

cluster_features = [
    'vv_mean', 'vv_std_temp', 'vh_mean', 'vvvh_mean', 'vvvh_std',
    'bsi_mean', 'bsi_std', 'ndvi_mean', 'ndvi_seasonal_amp',
    'vv_slope_year', 'bsi_slope_year', 'ndvi_slope_year',
]
Xc = StandardScaler().fit_transform(road_feats[cluster_features].values)
gm = GaussianMixture(n_components=3, random_state=42, covariance_type='full', n_init=5)
road_feats['cluster'] = gm.fit_predict(Xc)

# Label clusters by mean vv_mean (lowest VV = smoothest = good)
cluster_vv = road_feats.groupby('cluster')['vv_mean'].mean().sort_values()
cluster_to_class = dict(zip(cluster_vv.index, ['good', 'med', 'low']))
road_feats['cluster_label'] = road_feats['cluster'].map(cluster_to_class)

print('Cluster-derived distribution (%):')
print(road_feats['cluster_label'].value_counts(normalize=True).mul(100).round(1))

agreement = (road_feats['pred_condition'] == road_feats['cluster_label']).mean()
print(f'\nAgreement between physics score and clustering: {agreement:.1%}')

print('\nCross-tab (rows=physics-score, cols=cluster):')
print(pd.crosstab(road_feats['pred_condition'], road_feats['cluster_label'], normalize='index').round(2))

# Per-cluster signatures
print('\nCluster centroids (mean values):')
centroid_view = road_feats.groupby('cluster_label')[cluster_features].mean().round(3)
display(centroid_view)

## 8) Sanity check against manual validation (held-out)

If `data/named_roads.xlsx` exists with the `manual validation` sheet, compare the unsupervised labels to the manual ground truth. **This is not used for training** — purely a held-out evaluation.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

manual_path = DATA_DIR / 'named_roads.xlsx'
if not manual_path.exists():
    print(f'No manual validation file at {manual_path} — skipping check')
else:
    try:
        manual = pd.read_excel(manual_path, sheet_name='manual validation')
    except Exception as e:
        print(f'Could not read manual sheet: {e}')
        manual = None

    if manual is not None:
        manual['osm_id'] = manual['osm_id'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        label_col = None
        for col in ['pred_condition', 'manual_condition', 'condition', 'label']:
            if col in manual.columns:
                label_col = col
                break
        if label_col is None:
            print('No recognized label column. Columns present:', list(manual.columns)[:20])
        else:
            print(f'Using manual label column: {label_col!r}')
            eval_df = road_feats.merge(
                manual[['osm_id', label_col]].rename(columns={label_col: 'manual_label'}),
                on='osm_id', how='inner'
            )
            eval_df['manual_label'] = eval_df['manual_label'].astype(str).str.lower().str.strip()
            eval_df = eval_df[eval_df['manual_label'].isin(['good', 'med', 'low'])]
            print(f'Manual-set overlap: {len(eval_df)} roads')
            print('Manual class balance (%):')
            print(eval_df['manual_label'].value_counts(normalize=True).mul(100).round(1))

            if len(eval_df) >= 30:
                print('\n=== Physics-score vs manual ===')
                print(classification_report(eval_df['manual_label'], eval_df['pred_condition'],
                                            labels=['good', 'med', 'low'], digits=3, zero_division=0))

                print('\n=== Cluster-derived vs manual ===')
                print(classification_report(eval_df['manual_label'], eval_df['cluster_label'],
                                            labels=['good', 'med', 'low'], digits=3, zero_division=0))

                fig, axes = plt.subplots(1, 2, figsize=(10, 4))
                for ax, pred_col, title in zip(
                    axes,
                    ['pred_condition', 'cluster_label'],
                    ['Physics score', 'GMM cluster'],
                ):
                    cm = confusion_matrix(eval_df['manual_label'], eval_df[pred_col], labels=['good', 'med', 'low'])
                    ConfusionMatrixDisplay(cm, display_labels=['good', 'med', 'low']).plot(ax=ax, cmap='Blues', colorbar=False)
                    ax.set_title(f'{title} vs manual')
                plt.tight_layout()
                plt.show()
            else:
                print('Overlap too small (<30 roads) for meaningful report.')

## 9) Export predictions

In [ ]:
out_dir = ROOT / 'outputs'
out_dir.mkdir(exist_ok=True)
out_path = out_dir / 'layer2_new_predictions.csv'

export_cols = [
    'osm_id',
    'pred_condition', 'cluster_label',
    'combined_score', 'condition_score', 'deterioration_index',
    'vv_mean', 'vv_std_temp', 'vv_slope_year',
    'bsi_mean', 'bsi_slope_year',
    'ndvi_mean', 'ndvi_seasonal_amp', 'ndvi_slope_year',
]
export = road_feats[[c for c in export_cols if c in road_feats.columns]].copy()
export.to_csv(out_path, index=False)

print(f'Wrote {len(export)} predictions to {out_path}')
print('\nFinal class distribution:')
print(export['pred_condition'].value_counts())

## Notes & next steps

**Calibration:** The score weights (1.0 / 0.8 / 0.5 / 0.3 for condition; 1.0 / 0.5 / 0.3 for trend; 0.5 penalty on combination) are priors. If the manual-validation report in §8 shows systematic miscalls (e.g. too many `med` calls), the immediate dial to turn is the tercile threshold — replace with fixed values from the manual-set quantiles.

**Calibrating against an IRI proxy:** With even 50-100 roads of known IRI (Ghana Highway Authority surveys, World Bank ROCKS, eRoad), fit a simple linear model `IRI ~ combined_score + vv_mean + vv_slope_year` and use the residuals to rebalance weights. That would convert the current ordinal score into a calibrated IRI estimate.

**Layer 1 + Layer 2 integration:** This notebook uses OSM-tagged paved roads (~14k). The production path is to take Layer 1 predicted-paved (all roads classified paved by the XGBoost in `layer1_s1.ipynb`), then apply this scoring. Unpaved → `high` risk by policy rule, as in the original Layer 2.